# Interactive Companion: PyTorch Deep Dive
### Course: Machine Learning for Natural Language Processing 1 (UZH)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/lectures_supplementary_code_samples/l04_pytorch_companion.ipynb)

---

## 🎯 Didactic Motivation & Learning Objectives

In the lecture slides (`pytorch.tex`), we established a foundational conceptual bridge:
> **PyTorch does not introduce any new learning principles—it implements the gradient-based learning loop using batched tensors and modules.**

This notebook is an interactive companion designed to help you build an exact mental model of PyTorch. Rather than passively reading code, you will **predict**, **execute**, and **intentionally break** components to see how PyTorch works under the hood.

### What you will explore:
1. **Tensors & The Batch Dimension:** Predict tensor shapes, master broadcasting, and avoid the silent outer-product bug.
2. **Models as Modules:** Bridge column-vector textbook math ($h = Wx + b$) to batched row-vector code ($H = XW^\top + b$).
3. **Logits & Stable Losses:** Understand why networks output raw logits and why $\sigma(z)$ and $\mathrm{softmax}$ belong inside the loss.
4. **The Training Loop Anatomy:** Step through the 5-part inner loop and experimentally verify why `optimizer.zero_grad()` is indispensable.
5. **Capacity Investigation (Your Turn):** Empirically test whether 1 hidden ReLU unit can solve XOR across 20 random seeds and visualize why it fails geometrically.
6. **The Two Switches of Inference:** Disentangle `model.eval()` from `torch.no_grad()`.
7. **Scaling to FashionMNIST & Regularization:** Trace 4D image tensor transformations and test Dropout & Weight Decay.

---
## 0. Environment Setup & Hardware Acceleration
*(Appendix: Accelerators & Reproducibility)*

PyTorch tensors can run on hardware accelerators (NVIDIA CUDA, Apple MPS, or CPU).
Modern PyTorch provides `torch.accelerator` as a unified interface.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Ensure reproducibility across seeds
torch.manual_seed(42)
np.random.seed(42)

# Unified accelerator device selection (Slide 23)
if hasattr(torch, "accelerator") and torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type
elif torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"Active device:   {device}")

---
## 1. Tensors, Shapes, and the Batch Dimension
*(Slide 3: Scalars, Vectors, Matrices, Tensors)*

A PyTorch tensor is an $n$-dimensional array described by three attributes:
- **Shape:** Length along each dimension (e.g., `(B, F)` = batch size $\times$ feature count).
- **Data type (`dtype`):** Usually `torch.float32` for neural activations and parameters.
- **Device:** Memory location (`cpu`, `cuda`, `mps`).

Unlike a NumPy array, a PyTorch tensor can live on an accelerator and record computational graphs for autograd.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("Tensor x:\n", x)
print("Shape: ", x.shape)   # torch.Size([2, 3]) -> 2 rows (examples), 3 columns (features)
print("Dtype: ", x.dtype)   # torch.float32
print("Device:", x.device)  # cpu

---
## 2. Broadcasting Rules & The Bias Addition
*(Slides 4--6: Broadcasting Rules & Adding a Bias to a Batch)*

Broadcasting lets a small tensor act on a large one without copying memory.
PyTorch compares dimensions **from right to left**:
1. Equal sizes match.
2. A dimension of size 1 is stretched to match the other size.
3. A missing dimension counts as 1.
4. Anything else raises a `RuntimeError`.

### Adding a Bias to a Minibatch
In $H + b$, $H$ has shape `(B, F)` and $b$ has shape `(F,)`. The bias is broadcast across all batch rows.

In [ ]:
B, F = 8, 4
H = torch.zeros(B, F)              # Shape (8, 4)
b = torch.tensor([1., 2., 3., 4.])  # Shape (4,)

out = H + b
print(f"(8, 4) + (4,) -> Result shape: {out.shape}")
print("First two rows of H + b:\n", out[:2])

### 🧠 Interactive Activity: Predict the Shape (Slide 6)
Before running the code below, write down your prediction for each addition:
- **(A)** `(8, 4) + (4,)`
- **(B)** `(8, 4) + (8, 1)`
- **(C)** `(8, 4) + (8,)`
- **(D)** `(3, 1) + (1, 5)`

In [ ]:
shapes_to_test = [
    ("(A)", (8, 4), (4,)),
    ("(B)", (8, 4), (8, 1)),
    ("(C)", (8, 4), (8,)),
    ("(D)", (3, 1), (1, 5)),
]

for label, a, c in shapes_to_test:
    try:
        res = torch.zeros(a) + torch.zeros(c)
        print(f"{label} {a} + {c} -> {tuple(res.shape)}")
    except RuntimeError as e:
        print(f"{label} {a} + {c} -> ERROR: {e}")

### ⚠️ The Classic Deep Learning Silent Bug
Look at why **(C)** failed: PyTorch aligns from the right, so `(8,)` is compared with `4`, which mismatches!
To apply a per-example scalar weight across `(8, 4)`, it **must** have shape `(8, 1)`.

Now observe what happens if a binary prediction has shape `(B, 1)` and targets have shape `(B,)`:

In [ ]:
# Silent Bug Experiment
pred_col = torch.tensor([[0.7], [0.3], [0.9], [0.1]])  # Shape (4, 1)
target_row = torch.tensor([1.0, 0.0, 1.0, 0.0])        # Shape (4,)  <-- missing dimension!

# Subtracting (4, 1) and (4,) triggers outer broadcasting:
diff = pred_col - target_row
print(f"Shape of (pred_col - target_row): {diff.shape}")
print("Instead of 4 per-example errors, we created a 4x4 matrix of 16 cross-terms!")
print("Fix: always keep target shape as (B, 1) or call target.unsqueeze(1).")

---
## 3. Inside `nn.Linear`: From $Wx+b$ to $XW^\top+b$
*(Slides 7--9: From Wx+b to XW+b, Parameters)*

In textbooks:
$$h = Wx + b \quad (x \in \mathbb{R}^{d_{\text{in}}}, W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}})$$

In PyTorch, a batch $X \in \mathbb{R}^{B \times d_{\text{in}}}$ has examples as **rows**:
$$H = X W^\top + b$$
Therefore, `nn.Linear(d_in, d_out).weight` has shape **`(d_out, d_in)`**!

In [ ]:
lin = nn.Linear(in_features=2, out_features=4)

print(f"lin.weight.shape: {lin.weight.shape} (d_out, d_in)")
print(f"lin.bias.shape:   {lin.bias.shape}   (d_out,)")

# Verify the mathematical equivalence
X_batch = torch.tensor([[0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])  # (3, 2)
module_out = lin(X_batch)
math_out = X_batch @ lin.weight.T + lin.bias

print("Does lin(X) equal X @ W.T + b?")
print(torch.allclose(module_out, math_out))

---
## 4. The XOR Network & Logits with Stable Loss
*(Slides 7 & 10: The XOR Network as nn.Module, Logits and a Stable Loss)*

A model built with `nn.Module`:
- Creates layers & parameters in `__init__`.
- Computes representations in `forward`.
- **Returns raw, unnormalized logits $z$**.

### Why No Sigmoid in `forward`?
Computing $\log \sigma(z)$ directly inside `nn.BCEWithLogitsLoss` avoids catastrophic underflow $\log(0) = -\infty$ for extreme logit values.

In [ ]:
class XorMLP(nn.Module):
    def __init__(self, hidden=4):
        super().__init__()
        self.hidden = nn.Linear(2, hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        h = torch.relu(self.hidden(x))
        return self.out(h)  # returns raw logit

model = XorMLP(hidden=4)

print("Registered parameters in model:")
for name, p in model.named_parameters():
    print(f"  {name:15s} | shape: {tuple(p.shape)}")

---
## 5. The Training Loop in Action
*(Slides 11--12: Datasets, DataLoaders, The Training Loop)*

Here is the complete learning loop mapped directly to the slide diagram:
- `xb, yb in loader` $\to$ minibatch
- `model(xb)` $\to$ forward
- `loss_fn(...)` $\to$ loss
- `loss.backward()` $\to$ backprop & gradient computation
- `optimizer.step()` $\to$ parameter update
- `optimizer.zero_grad()` $\to$ clear accumulated gradients

In [ ]:
# XOR Dataset
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])  # Shape (4, 1) matches logit output

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

torch.manual_seed(0)
model = XorMLP(hidden=4)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Training loop
loss_history = []
for epoch in range(2000):
    epoch_loss = 0.0
    for xb, yb in loader:
        logits = model(xb)             # Forward
        loss = loss_fn(logits, yb)     # Loss
        optimizer.zero_grad()          # Clear gradients
        loss.backward()                # Backward
        optimizer.step()               # Step
        epoch_loss += loss.item() * len(xb)
    loss_history.append(epoch_loss / len(dataset))

print(f"Final training loss: {loss_history[-1]:.4f}")

### Visualizing the XOR Decision Boundary
Let us plot the model predictions across the 2D plane to verify that XOR is solved.

In [ ]:
def plot_decision_boundary(model_to_plot, title):
    gx, gy = torch.meshgrid(torch.linspace(-0.2, 1.2, 200),
                            torch.linspace(-0.2, 1.2, 200), indexing='xy')
    grid = torch.stack([gx.flatten(), gy.flatten()], dim=1)

    model_to_plot.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model_to_plot(grid)).reshape(gx.shape)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    cs = ax.contourf(gx.numpy(), gy.numpy(), probs.numpy(), levels=20, cmap="coolwarm", alpha=0.7)
    plt.colorbar(cs, label="P(y=1) = σ(z)")
    ax.contour(gx.numpy(), gy.numpy(), probs.numpy(), levels=[0.5], colors="black", linewidths=2)

    for pt, label in zip(X, y):
        color = "#C0392B" if label.item() == 1.0 else "#2980B9"
        marker = "o" if label.item() == 1.0 else "s"
        ax.scatter(pt[0].item(), pt[1].item(), c=color, marker=marker, s=130, edgecolors="k", zorder=5)

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.show()

plot_decision_boundary(model, "XorMLP (hidden=4) Decision Boundary")

---
## 6. Investigation: Can `hidden=1` Solve XOR?
*(Slide 13: Your Turn: One Hidden Unit)*

> **Geometric Prediction:**
> Each hidden ReLU unit produces one linear hyperplane bend. XOR requires isolating two opposite corners, which geometrically demands at least **two** units.
>
> Let us test this empirically across **20 random seeds** as presented in the lecture!

In [ ]:
solved_runs = 0
total_runs = 20

print(f"Testing XorMLP(hidden=1) across {total_runs} seeds...")
for s in range(total_runs):
    torch.manual_seed(s)
    m1 = XorMLP(hidden=1)
    o1 = torch.optim.SGD(m1.parameters(), lr=0.1)
    for _ in range(2000):
        for xb, yb in loader:
            l = loss_fn(m1(xb), yb)
            o1.zero_grad()
            l.backward()
            o1.step()
    with torch.no_grad():
        pred = (m1(X) > 0).float()
        if (pred == y).all():
            solved_runs += 1

print(f"=> Result: hidden=1 solved XOR in {solved_runs} / {total_runs} runs!")
plot_decision_boundary(m1, "Attempted Solution with hidden=1 (Single Linear Cut)")

---
## 7. Deep Dive: What Happens Without `zero_grad()`?
*(Slide 12: Training Loop & Gradient Accumulation)*

In PyTorch, `loss.backward()` **accumulates** (adds) to `.grad` instead of overwriting it (`param.grad += dL/dparam`).
If you forget `zero_grad()`, gradients keep compounding across iterations!

In [ ]:
test_m = XorMLP(hidden=4)
xb, yb = next(iter(loader))

print("Step-by-step gradient norm without zero_grad():")
for step in range(1, 6):
    loss = loss_fn(test_m(xb), yb)
    loss.backward()  # Accumulates!
    grad_norm = test_m.hidden.weight.grad.norm().item()
    print(f"  Step {step}: hidden.weight.grad L2 norm = {grad_norm:.4f}")

opt_test = torch.optim.SGD(test_m.parameters(), lr=0.1)
opt_test.zero_grad()
print(f"\nAfter calling zero_grad(): grad = {test_m.hidden.weight.grad}")

---
## 8. The Two Switches of Inference: `eval()` vs. `no_grad()`
*(Slide 14: Evaluation)*

- `model.train()` vs `model.eval()`: Controls **layer behavior** (Dropout, BatchNorm).
- `torch.no_grad()`: Controls **gradient tracking and memory**.

In [ ]:
drop_module = nn.Sequential(nn.Linear(10, 10), nn.Dropout(p=0.5))
sample_input = torch.ones(1, 10)

print("1. In train() mode (Stochastic: random units zeroed each pass):")
drop_module.train()
for i in range(3):
    out = drop_module(sample_input)
    print(f"   Pass {i+1} active units: {(out != 0).sum().item()} / 10")

print("\n2. In eval() mode (Deterministic: no dropout):")
drop_module.eval()
for i in range(3):
    out = drop_module(sample_input)
    print(f"   Pass {i+1} active units: {(out != 0).sum().item()} / 10")

---
## 9. Scaling Up to FashionMNIST & Regularization
*(Slides 15--18: FashionMNIST, The Tutorial Network, Regularization)*

Let us scale from 2D coordinates to $(1, 28, 28)$ images across 10 clothing classes.
We trace tensor shapes dynamically through the layers to verify the 669,706 parameter architecture.

In [ ]:
class FashionClassifier(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.flatten = nn.Flatten()
        if dropout_p > 0:
            self.net = nn.Sequential(
                nn.Linear(28*28, 512), nn.ReLU(), nn.Dropout(p=dropout_p),
                nn.Linear(512, 512), nn.ReLU(), nn.Dropout(p=dropout_p),
                nn.Linear(512, 10)
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(28*28, 512), nn.ReLU(),
                nn.Linear(512, 512), nn.ReLU(),
                nn.Linear(512, 10)
            )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

# Trace shapes
fashion_net = FashionClassifier()
batch_sample = torch.randn(64, 1, 28, 28)

print("Tracing tensor shapes:")
h = batch_sample
print(f"  Input:   {tuple(h.shape)}")
h = fashion_net.flatten(h)
print(f"  Flatten: {tuple(h.shape)}")
for idx, layer in enumerate(fashion_net.net):
    h = layer(h)
    print(f"  Layer {idx} ({layer.__class__.__name__:10s}): {tuple(h.shape)}")

param_count = sum(p.numel() for p in fashion_net.parameters())
print(f"\nTotal parameters: {param_count:,} (matches slide 16: 669,706)")

### Loading Data & Running Train / Test Loops
We load FashionMNIST using the current `v2` transforms pipeline and compare:
1. **Baseline Model:** SGD without regularization.
2. **Regularized Model:** SGD with Dropout ($p=0.2$) and Weight Decay (L2 penalty $\lambda = 10^{-4}$).

In [ ]:
from torchvision import datasets
from torchvision.transforms import v2

tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
train_ds = datasets.FashionMNIST(root="data", train=True, download=True, transform=tf)
test_ds = datasets.FashionMNIST(root="data", train=False, download=True, transform=tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

def train_loop(loader, model, loss_fn, optimizer, dev):
    model.train()
    total_loss, correct = 0.0, 0
    total = len(loader.dataset)
    for X, y in loader:
        X, y = X.to(dev), y.to(dev)
        logits = model(X)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X)
        correct += (logits.argmax(1) == y).sum().item()
    return total_loss / total, correct / total

def test_loop(loader, model, loss_fn, dev):
    model.eval()
    total_loss, correct = 0.0, 0
    total = len(loader.dataset)
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(dev), y.to(dev)
            logits = model(X)
            total_loss += loss_fn(logits, y).item() * len(X)
            correct += (logits.argmax(1) == y).sum().item()
    return total_loss / total, correct / total

# Run 3 training epochs for comparison
epochs = 3
ce_loss = nn.CrossEntropyLoss()

# 1. Baseline
m_base = FashionClassifier(dropout_p=0.0).to(device)
opt_base = torch.optim.SGD(m_base.parameters(), lr=0.05)
print("Training Baseline Model:")
for ep in range(epochs):
    tr_l, tr_a = train_loop(train_loader, m_base, ce_loss, opt_base, device)
    te_l, te_a = test_loop(test_loader, m_base, ce_loss, device)
    print(f"  Epoch {ep+1:2d} | Train Acc: {tr_a*100:.2f}% | Test Acc: {te_a*100:.2f}%")

# 2. Regularized
m_reg = FashionClassifier(dropout_p=0.2).to(device)
opt_reg = torch.optim.SGD(m_reg.parameters(), lr=0.05, weight_decay=1e-4)
print("\nTraining Regularized Model (Dropout 0.2 + Weight Decay 1e-4):")
for ep in range(epochs):
    tr_l, tr_a = train_loop(train_loader, m_reg, ce_loss, opt_reg, device)
    te_l, te_a = test_loop(test_loader, m_reg, ce_loss, device)
    print(f"  Epoch {ep+1:2d} | Train Acc: {tr_a*100:.2f}% | Test Acc: {te_a*100:.2f}%")

---
## 10. Self-Check & Review Questions
*(Slide 22: Questions)*

Review these questions before leaving the notebook:
1. **Which line of the training loop corresponds to which step of the learning loop?**
   - Minibatch $\to$ `for X, y in loader:`
   - Forward pass $\to$ `logits = model(X)`
   - Loss computation $\to$ `loss = loss_fn(logits, y)`
   - Backward pass $\to$ `loss.backward()`
   - Step $\to$ `optimizer.step()`
2. **Why does `nn.Linear(784, 512).weight` have shape `(512, 784)`?**
   - Because mathematical column vectors use $W x + b$ where $W \in \mathbb{R}^{512 \times 784}$. Stacking examples as rows computes $X W^\top + b$.
3. **What is the shape of `torch.zeros(16, 3) + torch.zeros(3)`?**
   - `(16, 3)`. Missing dimensions count as 1 and are broadcast.
4. **Why should `forward` return logits and not probabilities?**
   - Numerical stability in $\log \sigma(z)$ and $\log \mathrm{softmax}(z)$.
5. **What goes wrong if you evaluate a model with dropout without calling `eval()`?**
   - Neurons remain randomly dropped during test time, adding unnecessary noise and variance.
6. **Why do we call `zero_grad()`?**
   - To prevent gradient accumulation across multiple mini-batch updates.